# Does AML-specific adaptation fix foundation-model false positives after treatment?

**Question.** Does continued training of scGPT on adult AML cells reduce *false-positive malignant calls among non-malignant progenitors* (especially post-treatment ones) at a fixed detection sensitivity, compared with the original scGPT, HVG-PCA, scVI, and the same architecture trained from scratch?

**Data.** van Galen et al., *Cell* 2019 (GEO [GSE116256](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE116256)): 41 samples (AML diagnosis and post-treatment time points, plus healthy marrow), per-cell mutant/wild-type transcript counts. The seven-patient Ennis cohort is **not** used here; it belongs in a separate stress-test notebook, with the checkpoint saved by the last cell.

## Design (fixed before any result)
| Element | Choice |
|---|---|
| Evaluation unit | **patient** (patient-grouped K-fold; adaptation is re-run inside each fold on *training patients only*, unlabeled) |
| Task | cell-level malignant vs non-malignant among progenitor-compartment cells (HSC/Prog/GMP and their `-like` counterparts) |
| Probe | identical regularised logistic regression on every representation; C chosen by inner patient-grouped CV |
| Operating point | threshold set on *training* patients to give 90% sensitivity; never tuned on the test fold |
| **Primary endpoint** | false-positive rate in **post-treatment non-malignant progenitors**, patient-macro-averaged, 95% patient-cluster bootstrap CI |
| Contrasts | C1 adapted vs original scGPT · C2 vs PCA · C3 vs scVI · C4 vs from-scratch (does pretraining matter?) |
| Pass rule | CI of the FPR difference entirely below 0 **and** sensitivity within 5 points of the comparator |
| Abandon rule | if C1–C3 do not all pass, drop the model-development claim |

## What the feasibility gate showed (computed on the real files before building this)
* Post-treatment progenitors the original classifier calls normal: **453 cells, but only 4 patients with ≥10 cells**; one patient (AML556) supplies ~39%.
* Of those, only 145 cells (3 patients) have any wild-type read coverage, and wild-type-only is a *weak* normal label (heterozygous leukemic cells with allele dropout look the same).
* Genotype-confirmed malignant post-treatment progenitors: **28 cells from 2 patients**.

So this cohort **cannot support a confirmatory claim**. Cell 4 recomputes the gate and the final cell labels the verdict *exploratory* if it is not met. A real test needs a cohort with independent per-cell labels in post-treatment marrow (e.g. donor/recipient genotype demultiplexing after transplant, or CNV-defined malignant cells), which is the thing to look for next.

## Other limits to keep in mind
* Labels are van Galen's random-forest calls, not independent truth; expression-based probes agreeing with them is partly circular. `geno == MUT` cells are the genotype-supported subset.
* Adaptation here is masked-value prediction (GEP + GEPC, mask ratios 0.25/0.5/0.75). The original scGPT also used a generative objective; this is an approximation of its pretraining.
* Not a clinical MRD assay; nothing here estimates MRD sensitivity.

**Runtime (T4):** ~10–15 min per adaptation × 5 folds × (adapted + scratch) plus scVI ≈ 2–3 h. Set `RUN_SCRATCH=False` to halve it. `SMOKE_TEST=True` in cell 2 runs a synthetic 2–3 minute plumbing check. Results and caches live on Google Drive and runs resume after disconnects.


In [ ]:
#@title 1 · Install prerequisites (Colab)
# Same pinned stack that ran your zero-shot notebook successfully.
%pip -q install "anndata>=0.10" "scanpy>=1.10" "huggingface_hub>=0.25" "transformers>=4.40,<4.49" "scgpt==0.2.4" "scikit-learn" "pandas" "matplotlib" "seaborn" "tqdm" "requests"

In [ ]:
#@title 2 · Configuration (everything tunable lives here)
import os, sys, re, io, json, math, time, gzip, shutil, warnings, itertools
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp
import torch
warnings.filterwarnings('ignore')

SEED = 42
SMOKE_TEST = False       # True = synthetic data + tiny random "pretrained" model: a 2-3 min plumbing check, NOT science
USE_DRIVE  = True        # keep downloads/caches on Google Drive so a runtime disconnect doesn't lose work
IN_COLAB   = 'google.colab' in sys.modules
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- design (pre-specified; change BEFORE looking at results, not after) ----
N_FOLDS         = 5      # patient-grouped outer folds; adaptation is re-run inside each fold on TRAINING patients only
TARGET_TPR      = 0.90   # fixed detection sensitivity; threshold is set on training patients, never on the test fold
PROGENITOR_TYPES = ['HSC','Prog','GMP','HSC-like','Prog-like','GMP-like']
MIN_CELLS_PATIENT_STRATUM = 10   # a patient contributes to a stratum's macro-average only with >= this many cells
N_BOOT = 5000            # patient-cluster bootstrap replicates
C_GRID = (0.01, 0.1, 1.0)        # logistic-probe regularisation, chosen by INNER patient-grouped CV
PROBE_MAX_PER_SAMPLE_LABEL = 300 # stops one huge sample (e.g. AML328-D29) dominating probe training

# ---- scGPT adaptation ----
MAX_LEN        = 1200    # same as your zero-shot run
ADAPT_STEPS    = 1500    # optimizer steps per fold (~1 pass over a fold's training cells at batch 16)
ADAPT_BATCH    = 8       # micro-batch; lower if a T4 runs out of memory
ADAPT_ACCUM    = 2       # effective batch = 16
ADAPT_LR       = 2e-5    # continued training of pretrained weights (original pretraining used 1e-4)
SCRATCH_LR     = 1e-4    # same architecture, random init
MASK_RATIOS    = (0.25, 0.5, 0.75)  # original scGPT pretraining mask ratios
MAX_ADAPT_CELLS_PER_SAMPLE = 1500
RUN_SCRATCH    = True    # ablation: same architecture + same AML-only objective, NO foundation-model pretraining
RUN_SCVI       = True    # conventional baseline #2 (skipped gracefully if scvi-tools is unusable)
TRAIN_FINAL_MODEL = True # last cell: adapt on ALL patients and save a checkpoint for the seven-patient follow-up notebook

if SMOKE_TEST:
    N_FOLDS, ADAPT_STEPS, ADAPT_BATCH, ADAPT_ACCUM, N_BOOT = 3, 30, 4, 1, 300
    MAX_LEN, MIN_CELLS_PATIENT_STRATUM, TRAIN_FINAL_MODEL = 200, 3, False

if IN_COLAB and USE_DRIVE and not SMOKE_TEST:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AML_FM_adaptation')
elif IN_COLAB:
    ROOT = Path('/content/AML_FM_adaptation')
else:
    ROOT = Path.cwd()/'AML_FM_adaptation'
if SMOKE_TEST: ROOT = ROOT.parent/'AML_FM_adaptation_SMOKE'
DATA, OUT, CACHE, MODEL = ROOT/'data', ROOT/'results', ROOT/'cache', ROOT/'models'
FIG = OUT/'figures'
for d in (DATA, OUT, CACHE, MODEL, FIG): d.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 0 if SMOKE_TEST else min(2, os.cpu_count() or 1)
print('device:', DEVICE, '| root:', ROOT, '| smoke:', SMOKE_TEST)
if DEVICE == 'cuda': print('GPU:', torch.cuda.get_device_name(0))
else: print('WARNING: no GPU. Adaptation will be impractically slow; use Runtime > Change runtime type > T4/L4.')

In [ ]:
#@title 3 · Download van Galen 2019 (GSE116256) per-sample count + annotation files
import requests, urllib.request
from tqdm.auto import tqdm
GEO_TEXT = 'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE116256&targ=all&form=text&view=brief'
MAN = DATA/'GSE116256_manifest.json'
SKIP_SAMPLES = {'MUTZ3', 'OCI-AML3'}          # cell lines, not patients

def fetch(url, dest, tries=12):
    """Resumable download (HTTP Range) with backoff; verifies the gzip stream before saving."""
    if dest.exists(): return
    buf, total = bytearray(), None
    for a in range(1, tries+1):
        try:
            req = urllib.request.Request(url, headers={'Range': f'bytes={len(buf)}-'} if buf else {})
            with urllib.request.urlopen(req, timeout=120) as r:
                if buf and r.status != 206: buf = bytearray()          # server ignored Range -> restart cleanly
                total = total or (int(r.headers['Content-Length']) + len(buf) if r.headers.get('Content-Length') else None)
                while True:
                    c = r.read(1 << 18)
                    if not c: break
                    buf += c
            if total and len(buf) != total: raise IOError('short read')
            gzip.decompress(bytes(buf))                                  # integrity check: raises on truncated gzip
            dest.write_bytes(bytes(buf)); return
        except Exception as e:
            print(f'  retry {a}/{tries} {dest.name}: {type(e).__name__} (have {len(buf)} bytes)'); time.sleep(min(30, 2*a))
    raise RuntimeError(f'could not download {url}')

if SMOKE_TEST:
    print('SMOKE_TEST: skipping download'); manifest = {}
else:
    if not MAN.exists():
        r = requests.get(GEO_TEXT, timeout=180); r.raise_for_status()
        files = {}
        for line in r.text.splitlines():
            if re.match(r'!Sample_supplementary_file_\d+ =', line):   # healthy donors list the .dem as file_2
                url = line.split('=', 1)[1].strip().replace('ftp://', 'https://')
                m = re.match(r'GSM\d+_(.+)\.(dem|anno)\.txt\.gz$', url.rsplit('/', 1)[1])
                if m and m.group(1) not in SKIP_SAMPLES: files.setdefault(m.group(1), {})[m.group(2)] = url
        MAN.write_text(json.dumps({k: v for k, v in files.items() if len(v) == 2}))
    manifest = json.loads(MAN.read_text())
    assert len(manifest) >= 41, f'unexpected manifest size {len(manifest)}'
    (DATA/'dem').mkdir(exist_ok=True); (DATA/'anno').mkdir(exist_ok=True)
    for s, d in tqdm(sorted(manifest.items()), desc='samples'):
        fetch(d['anno'], DATA/'anno'/f'{s}.anno.txt.gz'); fetch(d['dem'], DATA/'dem'/f'{s}.dem.txt.gz')
    print(len(manifest), 'samples ready')

In [ ]:
#@title 4 · Annotation, genotype tiers, and the FEASIBILITY GATE (run this before believing anything downstream)
def count_transcripts(x):
    """'NPM1.W288fs/913 /// NPM1.W288fs/393' -> 1306 ; NaN -> 0  (van Galen stores gene.variant/count strings)."""
    if not isinstance(x, str): return 0
    t = 0
    for p in x.split('///'):
        m = re.search(r'/(\d+)\s*$', p.strip())
        if m: t += int(m.group(1))
    return t

def make_synthetic_ann(seed=0):
    r = np.random.default_rng(seed); rows = []
    ctypes = ['HSC-like','Prog-like','GMP-like','Mono-like','HSC','Prog','GMP','Mono','T','NK']
    spec = [(f'A{i}', ['D0', 'D20']) for i in range(1, 9)] + [(f'H{i}', [None]) for i in range(1, 4)]
    for pat, days in spec:
        for dd in days:
            samp = pat if dd is None else f'{pat}-{dd}'
            for j in range(160):
                healthy = dd is None; post = dd == 'D20'
                mal = (not healthy) and r.random() < (0.35 if post else 0.7)
                ct = r.choice(ctypes[:3] if r.random() < .6 else ctypes)
                if mal: ct = r.choice(['HSC-like','Prog-like','GMP-like','Mono-like'])
                elif ct.endswith('-like'): ct = ct[:-5]
                rows.append(dict(Cell=f'{samp}_{j}', sample=samp, MutTranscripts=('X.Y/5' if mal and r.random() < .1 else np.nan),
                                 WtTranscripts=('X.wt/5' if (not mal) and r.random() < .3 else np.nan),
                                 PredictionRefined='malignant' if mal else 'normal', CellType=ct))
    return pd.DataFrame(rows)

if SMOKE_TEST:
    ANN = make_synthetic_ann()
else:
    fr = []
    for s in sorted(manifest):
        a = pd.read_csv(DATA/'anno'/f'{s}.anno.txt.gz', sep='\t', dtype=str)
        a['sample'] = s; fr.append(a[['Cell','sample','MutTranscripts','WtTranscripts','PredictionRefined','CellType']])
    ANN = pd.concat(fr, ignore_index=True)

ANN['patient'] = ANN['sample'].str.extract(r'^(AML\d+[A-Z]?|BM\d+|A\d+|H\d+)')[0]
assert ANN.patient.notna().all(), ANN.loc[ANN.patient.isna(), 'sample'].unique()
day = ANN['sample'].str.extract(r'-D(\d+)$')[0].astype(float)
is_healthy = ANN.patient.str.match(r'^(BM|H)\d+$')
ANN['stage'] = np.where(is_healthy, 'healthy', np.where(day == 0, 'dx', 'post'))   # post = any sample after day 0
assert (is_healthy | day.notna()).all(), 'sample without a day suffix that is not a healthy donor'
ANN['mut'] = ANN.MutTranscripts.map(count_transcripts); ANN['wt'] = ANN.WtTranscripts.map(count_transcripts)
ANN['geno'] = np.where(ANN.mut > 0, 'MUT', np.where(ANN.wt > 0, 'WT-only', 'none'))
ANN['y'] = ANN.PredictionRefined.map({'malignant': 1.0, 'normal': 0.0})          # 'unclear' -> NaN (excluded)
ANN['prog'] = ANN.CellType.isin(PROGENITOR_TYPES)
ANN.to_csv(OUT/'00_annotation_all_cells.csv.gz', index=False)

P = ANN[ANN.prog & ANN.y.notna()]
print('Progenitor-compartment cells by stage x label x genotype tier:')
display(P.groupby(['stage', 'PredictionRefined', 'geno']).size().unstack(fill_value=0))

pn = P[(P.stage == 'post') & (P.y == 0)]
gate = dict(
    post_normal_cells = len(pn),
    post_normal_patients_ge_min = int((pn.groupby('patient').size() >= MIN_CELLS_PATIENT_STRATUM).sum()),
    post_normal_top_patient_share = float(pn.patient.value_counts(normalize=True).iloc[0]) if len(pn) else np.nan,
    post_normal_with_WT_coverage_cells = int((pn.geno == 'WT-only').sum()),
    post_normal_with_WT_coverage_patients = int(pn[pn.geno == 'WT-only'].patient.nunique()),
    post_malignant_MUT_cells = int(((P.stage == 'post') & (P.geno == 'MUT')).sum()),
    post_malignant_MUT_patients = int(P[(P.stage == 'post') & (P.geno == 'MUT')].patient.nunique()),
    healthy_progenitor_donors = int(P[P.stage == 'healthy'].patient.nunique()))
GATE = pd.Series(gate); display(GATE.to_frame('value'))
GATE.to_csv(OUT/'00_feasibility_gate.csv')
PRIMARY_CONFIRMATORY = gate['post_normal_patients_ge_min'] >= 5 and gate['post_normal_cells'] >= 100
print('\nPrimary stratum (post-treatment normal progenitors) adequately sized for a confirmatory claim:', PRIMARY_CONFIRMATORY)
print('''
READ THIS BEFORE INTERPRETING ANY RESULT
 * y ("malignant"/"normal") is van Galen's random-forest call (PredictionRefined), trained on mutation-detected cells.
   Our probes are expression-based, so agreement with y is NOT independent proof of leukemia status.
 * The only genotype-supported malignant cells are geno == "MUT". For the post-treatment stratum there are very few.
 * geno == "WT-only" means wild-type reads were seen but no mutant reads. That fits normal cells AND heterozygous
   leukemic cells with allele dropout, so it is a weak normal label, never proof.
 * "post" = any sample after day 0: it includes persistent/relapsing disease, not only remission marrow.
 * Healthy-donor progenitors are the only unambiguously non-malignant cells here, and they are not treatment-regenerating.
Verdict: with these counts, treat this notebook as a feasibility/pilot analysis unless the gate above is comfortably met.''')

In [ ]:
#@title 5 · Assemble the count matrix (cells x genes, raw UMIs)
import anndata as ad
H5 = DATA/'vangalen_counts.h5ad'
N_SYN_GENES = 600
if H5.exists() and not SMOKE_TEST:
    adata = ad.read_h5ad(H5)
else:
    if SMOKE_TEST:
        r = np.random.default_rng(1); genes = np.array([f'G{i:04d}' for i in range(N_SYN_GENES)])
        base = r.gamma(0.5, 1.0, N_SYN_GENES) * 0.8
        mat = np.zeros((len(ANN), N_SYN_GENES), np.float32)
        for i, (mal, pg) in enumerate(zip(ANN.y.values, ANN.prog.values)):
            lam = base.copy()
            if mal == 1: lam[:60] *= 4
            if pg: lam[60:120] *= 3
            mat[i] = r.poisson(lam)
        X = sp.csr_matrix(mat); adata = ad.AnnData(X, obs=ANN.set_index('Cell').copy(), var=pd.DataFrame(index=genes))
    else:
        blocks, gene_ref = [], None
        for s in tqdm(sorted(manifest), desc='reading counts'):
            df = pd.read_csv(DATA/'dem'/f'{s}.dem.txt.gz', sep='\t', index_col=0)
            if df.index.duplicated().any(): df = df.groupby(level=0).sum()
            if gene_ref is None: gene_ref = df.index
            else: df = df.reindex(gene_ref, fill_value=0)
            blocks.append(sp.csr_matrix(df.values.T.astype(np.float32))); blocks[-1].cell_ids = list(df.columns)
        X = sp.vstack(blocks, format='csr'); cells = sum([b.cell_ids for b in blocks], [])
        obs = ANN.drop_duplicates('Cell').set_index('Cell').reindex(cells)
        keep = obs.patient.notna().to_numpy() & (np.asarray(X.sum(1)).ravel() > 0)
        adata = ad.AnnData(X[keep], obs=obs[keep].copy(), var=pd.DataFrame(index=np.asarray(gene_ref, dtype=str)))
        adata.write_h5ad(H5)
assert adata.obs_names.is_unique
assert np.allclose(adata.X.data, np.rint(adata.X.data)), 'counts are not integers'
print(adata)
# Evaluation set: labelled progenitor-compartment cells. Adaptation corpus later uses ALL cells of TRAINING patients, unlabeled.
obs = adata.obs.copy()
obs['cell'] = obs.index
PROG_MASK = (obs.prog & obs.y.notna()).to_numpy()
PROG_IDX = np.flatnonzero(PROG_MASK)
print('labelled progenitor cells for evaluation:', len(PROG_IDX), 'of', adata.n_obs)

In [ ]:
#@title 6 · scGPT: checkpoint, embedding function, and continued-training (adaptation) loop
import importlib, types, scanpy as sc
import scgpt
from scgpt.model import TransformerModel
from scgpt.data_collator import DataCollator
from scgpt.loss import masked_mse_loss
from scgpt.utils import load_pretrained
from torch.utils.data import DataLoader, RandomSampler

class JSONGeneVocab:
    """Minimal vocab (torchtext-free) that keeps the checkpoint's EXACT token ids (same trick as your zero-shot notebook)."""
    def __init__(self, token_to_id):
        self._stoi = {str(k): int(v) for k, v in token_to_id.items()}
        if sorted(self._stoi.values()) != list(range(len(self._stoi))): raise ValueError('vocab ids must be consecutive')
    def __len__(self): return len(self._stoi)
    def __contains__(self, t): return t in self._stoi
    def __getitem__(self, t): return self._stoi[t]
    def append_token(self, t): self._stoi[t] = len(self._stoi)
    def get_stoi(self): return dict(self._stoi)

SCGPT_DIR = MODEL/'scGPT-human'
def prepare_checkpoint():
    SCGPT_DIR.mkdir(exist_ok=True)
    if SMOKE_TEST:      # tiny random "pretrained" checkpoint written in the real file format
        genes = [f'G{i:04d}' for i in range(N_SYN_GENES)]
        stoi = {t: i for i, t in enumerate(['<pad>', '<cls>', '<eoc>'] + genes)}
        cfg = dict(embsize=32, nheads=2, d_hid=32, nlayers=2, n_layers_cls=1, dropout=0.1, pad_token='<pad>', pad_value=-2, mask_value=-1)
        (SCGPT_DIR/'vocab.json').write_text(json.dumps(stoi)); (SCGPT_DIR/'args.json').write_text(json.dumps(cfg))
        torch.manual_seed(0); m, _ = build_scgpt(SCGPT_DIR, pretrained=False, device='cpu')
        torch.save(m.state_dict(), SCGPT_DIR/'best_model.pt')
    else:
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id='wanglab/scGPT-human', repo_type='model', allow_patterns=['args.json', 'vocab.json', 'best_model.pt'], local_dir=str(SCGPT_DIR))
    assert all((SCGPT_DIR/f).exists() for f in ('args.json', 'vocab.json', 'best_model.pt'))

def build_scgpt(model_dir, pretrained=True, device=DEVICE):
    """Mirror of scgpt.tasks.embed_data's model construction (non-flash attention), optionally with random init."""
    model_dir = Path(model_dir)
    vocab = JSONGeneVocab(json.loads((model_dir/'vocab.json').read_text()))
    for t in ['<pad>', '<cls>', '<eoc>']:
        if t not in vocab: vocab.append_token(t)
    cfg = json.loads((model_dir/'args.json').read_text())
    model = TransformerModel(ntoken=len(vocab), d_model=cfg['embsize'], nhead=cfg['nheads'], d_hid=cfg['d_hid'], nlayers=cfg['nlayers'],
                             nlayers_cls=cfg['n_layers_cls'], n_cls=1, vocab=vocab, dropout=cfg['dropout'], pad_token=cfg['pad_token'],
                             pad_value=cfg['pad_value'], do_mvc=True, do_dab=False, use_batch_labels=False, domain_spec_batchnorm=False,
                             explicit_zero_prob=False, use_fast_transformer=False, fast_transformer_backend='flash', pre_norm=False)
    if pretrained:
        sd = torch.load(model_dir/'best_model.pt', map_location='cpu')
        load_pretrained(model, sd, verbose=False)
        sd2 = {k.replace('Wqkv.', 'in_proj_'): v for k, v in sd.items()}
        ms = model.state_dict(); ok = sum(k in sd2 and sd2[k].shape == v.shape for k, v in ms.items())
        print(f'pretrained tensors loaded: {ok}/{len(ms)}')
        assert ok >= 0.9*len(ms), 'most tensors failed to load; do not interpret results'
    cfg.setdefault('mask_value', -1)
    return model.to(device), (vocab, cfg)

class CellDS(torch.utils.data.Dataset):
    """Sparse raw counts -> {<cls> + expressed genes (vocab ids), values}; binning/masking happen in scGPT's DataCollator."""
    def __init__(self, X, gene_ids, cls_id, pad_value):
        self.X, self.g, self.cls, self.pv = X.tocsr(), gene_ids, cls_id, pad_value
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i):
        s, e = self.X.indptr[i], self.X.indptr[i+1]
        genes = np.insert(self.g[self.X.indices[s:e]], 0, self.cls); vals = np.insert(self.X.data[s:e].astype(np.float32), 0, self.pv)
        return dict(id=i, genes=torch.from_numpy(genes).long(), expressions=torch.from_numpy(vals))

def vocab_map(var_names, vocab):
    ids = np.array([vocab[g] if g in vocab else -1 for g in var_names]); return ids, ids >= 0

@torch.no_grad()
def embed_cells(model, X, gene_ids, vocab, cfg, batch_size=32, seed=0):
    """Cell embedding = <cls> state, L2-normalised (identical recipe to scgpt.tasks.embed_data)."""
    torch.manual_seed(seed); model.eval(); pad_id = vocab[cfg['pad_token']]
    ds = CellDS(X, gene_ids, vocab['<cls>'], cfg['pad_value'])
    col = DataCollator(do_padding=True, pad_token_id=pad_id, pad_value=cfg['pad_value'], do_mlm=False, do_binning=True,
                       max_length=MAX_LEN, sampling=True, keep_first_n_tokens=1)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=col, num_workers=NUM_WORKERS)
    out, n = np.zeros((len(ds), cfg['embsize']), np.float32), 0
    dev = next(model.parameters()).device
    for b in tqdm(dl, desc='embedding', leave=False):
        g = b['gene'].to(dev)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=dev.type == 'cuda'):
            h = model._encode(g, b['expr'].to(dev), src_key_padding_mask=g.eq(pad_id))[:, 0, :]
        out[n:n+len(h)] = h.float().cpu().numpy(); n += len(h)
    assert np.isfinite(out).all(), 'non-finite embeddings'
    return out / np.linalg.norm(out, axis=1, keepdims=True).clip(1e-8)

class RandMaskCollator:
    """scGPT pretraining drew the mask ratio from {0.25,0.5,0.75}; do the same per batch."""
    def __init__(self, base, ratios): self.base, self.ratios = base, ratios
    def __call__(self, ex):
        self.base.mlm_probability = float(self.ratios[int(torch.randint(len(self.ratios), (1,)))]); return self.base(ex)

def _loss(model, b, dev, pad_id, mask_value):
    g, tgt, masked = b['gene'].to(dev), b['expr'].to(dev), b['masked_expr'].to(dev)
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=dev.type == 'cuda'):
        o = model(g, masked, src_key_padding_mask=g.eq(pad_id), MVC=True)
    pos = masked.eq(mask_value)
    return masked_mse_loss(o['mlm_output'].float(), tgt.float(), pos), masked_mse_loss(o['mvc_output'].float(), tgt.float(), pos)

@torch.no_grad()
def heldout_masked_loss(model, ds, vocab, cfg, n_batches=8, batch=8):
    """Fixed-seed masked-value loss on held-out cells of the TRAINING patients: a sanity check that adaptation learns."""
    torch.manual_seed(1234); model.eval(); dev = next(model.parameters()).device; pad_id = vocab[cfg['pad_token']]
    col = RandMaskCollator(DataCollator(do_padding=True, pad_token_id=pad_id, pad_value=cfg['pad_value'], do_mlm=True, do_binning=True,
                           mask_value=cfg['mask_value'], max_length=MAX_LEN, sampling=True, keep_first_n_tokens=1), (0.5,))
    dl = DataLoader(ds, batch_size=batch, shuffle=False, collate_fn=col); tot = []
    for i, b in enumerate(dl):
        if i >= n_batches: break
        a, c = _loss(model, b, dev, pad_id, cfg['mask_value']); tot.append((a + c).item())
    return float(np.mean(tot))

def adapt_scgpt(model, X, gene_ids, vocab, cfg, steps, lr, seed, label):
    """Continued masked-value training (GEP + GEPC) on UNLABELED cells. Patient labels/malignancy are never used."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed); dev = next(model.parameters()).device; pad_id = vocab[cfg['pad_token']]
    perm = rng.permutation(X.shape[0]); n_val = min(128, max(8, X.shape[0]//20))
    val_ds = CellDS(X[perm[:n_val]], gene_ids, vocab['<cls>'], cfg['pad_value'])
    tr_ds = CellDS(X[perm[n_val:]], gene_ids, vocab['<cls>'], cfg['pad_value'])
    base = DataCollator(do_padding=True, pad_token_id=pad_id, pad_value=cfg['pad_value'], do_mlm=True, do_binning=True,
                        mask_value=cfg['mask_value'], max_length=MAX_LEN, sampling=True, keep_first_n_tokens=1)
    gen = torch.Generator().manual_seed(seed)
    dl = DataLoader(tr_ds, batch_size=ADAPT_BATCH, sampler=RandomSampler(tr_ds, replacement=True, num_samples=steps*ADAPT_ACCUM*ADAPT_BATCH, generator=gen),
                    collate_fn=RandMaskCollator(base, MASK_RATIOS), num_workers=NUM_WORKERS, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    warm = max(1, int(0.05*steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s+1)/warm if s < warm else 0.1 + 0.9*0.5*(1+math.cos(math.pi*(s-warm)/max(1, steps-warm))))
    scaler = torch.amp.GradScaler('cuda', enabled=dev.type == 'cuda')
    l0 = heldout_masked_loss(model, val_ds, vocab, cfg); hist = []
    model.train(); step, micro, t0 = 0, 0, time.time()
    for b in tqdm(dl, total=steps*ADAPT_ACCUM, desc=f'adapt[{label}]', leave=False):
        a, c = _loss(model, b, dev, pad_id, cfg['mask_value'])
        scaler.scale((a + c)/ADAPT_ACCUM).backward(); micro += 1
        if micro % ADAPT_ACCUM == 0:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
            hist.append((a + c).item())
    l1 = heldout_masked_loss(model, val_ds, vocab, cfg)
    print(f'  [{label}] held-out masked loss (training patients): {l0:.4f} -> {l1:.4f} | {steps} steps, {(time.time()-t0)/60:.1f} min')
    model.eval(); return dict(label=label, loss_start=l0, loss_end=l1, train_loss=hist)

prepare_checkpoint()
model0, (VOCAB, CFG) = build_scgpt(SCGPT_DIR, pretrained=True)
GENE_IDS_ALL, IN_VOCAB = vocab_map(adata.var_names.astype(str), VOCAB)
print(f'scGPT vocabulary covers {IN_VOCAB.sum():,}/{len(IN_VOCAB):,} genes ({IN_VOCAB.mean():.1%})')
assert IN_VOCAB.mean() > 0.5 if not SMOKE_TEST else True
X_V = adata.X[:, np.flatnonzero(IN_VOCAB)].tocsr(); GENE_IDS = GENE_IDS_ALL[IN_VOCAB]   # scGPT-visible genes only

In [ ]:
#@title 7 · Patient-grouped folds (greedy balance so each fold holds post-treatment normal progenitors)
groups = obs.groupby('patient').apply(lambda d: pd.Series(dict(
    n_prog=int(d.index.isin(obs.index[PROG_IDX]).sum()),
    n_post_norm=int(((d.stage == 'post') & (d.y == 0) & d.prog).sum()),
    n_post_mal=int(((d.stage == 'post') & (d.y == 1) & d.prog).sum())))).reset_index()
groups['w'] = groups.n_prog + 10*groups.n_post_norm + 10*groups.n_post_mal
fold_w = np.zeros(N_FOLDS); FOLD_OF = {}
for _, r in groups.sort_values('w', ascending=False).iterrows():
    k = int(np.argmin(fold_w)); FOLD_OF[r.patient] = k; fold_w[k] += r.w
groups['fold'] = groups.patient.map(FOLD_OF); display(groups.sort_values(['fold', 'patient']))
FOLDS = [sorted(p for p, k in FOLD_OF.items() if k == f) for f in range(N_FOLDS)]
groups.to_csv(OUT/'01_folds.csv', index=False)

In [ ]:
#@title 8 · Original (frozen) scGPT embeddings of every labelled progenitor cell
f = CACHE/'emb_scGPT_zeroshot.npy'
if f.exists(): Z0 = np.load(f)
else:
    Z0 = embed_cells(model0, X_V[PROG_IDX], GENE_IDS, VOCAB, CFG, seed=SEED); np.save(f, Z0)
print('zero-shot embeddings', Z0.shape)
del model0
if DEVICE == 'cuda': torch.cuda.empty_cache()

In [ ]:
#@title 9 · Per-fold representations fitted on TRAINING patients only: PCA, scVI, AML-adapted scGPT, from-scratch scGPT
from sklearn.decomposition import PCA
SCVI_EPOCHS = 5 if SMOKE_TEST else 40
REPS = ['PCA'] + (['scVI'] if RUN_SCVI else []) + ['scGPT_zeroshot', 'scGPT_adapted'] + (['scGPT_scratch'] if RUN_SCRATCH else [])
EMB = {r: {} for r in REPS}; ADAPT_LOG = {}

def log_cp10k(X):
    lib = np.asarray(X.sum(1)).ravel().clip(1); Y = sp.diags(1e4/lib) @ X; Y = Y.tocsr(); Y.data = np.log1p(Y.data); return Y

def fit_hvg_pca(train_rows, n_hvg=3000, n_comp=50, seed=SEED):
    r = np.random.default_rng(seed); rows = r.choice(train_rows, size=min(len(train_rows), 20000), replace=False)
    Y = log_cp10k(adata.X[rows]); mu = np.asarray(Y.mean(0)).ravel(); var = np.asarray(Y.power(2).mean(0)).ravel() - mu**2
    hvg = np.argsort(var)[-min(n_hvg, Y.shape[1]):]
    D = Y[:, hvg].toarray(); center = D.mean(0)
    pca = PCA(n_components=min(n_comp, D.shape[1]-1, D.shape[0]-1), random_state=seed).fit(D - center)
    return hvg, center, pca

def adapt_corpus(train_rows, seed):
    r = np.random.default_rng(seed); keep = []
    for s, idx in obs.iloc[train_rows].groupby('sample').indices.items():
        keep.extend(train_rows[idx][r.permutation(len(idx))[:MAX_ADAPT_CELLS_PER_SAMPLE]])
    return np.sort(np.array(keep))

def run_scvi(train_rows, hvg, seed):
    import scvi
    scvi.settings.seed = seed
    r = np.random.default_rng(seed); rows = np.sort(r.choice(train_rows, size=min(len(train_rows), 30000), replace=False))
    a_tr = ad.AnnData(adata.X[rows][:, hvg].astype(np.float32)); scvi.model.SCVI.setup_anndata(a_tr)
    m = scvi.model.SCVI(a_tr, n_latent=30); m.train(max_epochs=SCVI_EPOCHS, early_stopping=False)
    a_p = ad.AnnData(adata.X[PROG_IDX][:, hvg].astype(np.float32))
    return np.asarray(m.get_latent_representation(a_p), dtype=np.float32)

for k in range(N_FOLDS):
    test = FOLDS[k]; train_rows = np.flatnonzero(~obs.patient.isin(test).to_numpy())
    print(f'\n=== fold {k}: held-out patients {test} | {len(train_rows):,} training cells ===')
    EMB['scGPT_zeroshot'][k] = Z0
    # --- PCA (+ shared HVG set for scVI) ---
    f = CACHE/f'fold{k}_PCA.npy'
    hvg, center, pca = fit_hvg_pca(train_rows)
    if f.exists(): EMB['PCA'][k] = np.load(f)
    else:
        D = log_cp10k(adata.X[PROG_IDX])[:, hvg].toarray() - center; EMB['PCA'][k] = pca.transform(D).astype(np.float32); np.save(f, EMB['PCA'][k])
    # --- scVI ---
    if RUN_SCVI:
        f = CACHE/f'fold{k}_scVI.npy'
        if f.exists(): EMB['scVI'][k] = np.load(f)
        else:
            try: EMB['scVI'][k] = run_scvi(train_rows, hvg, SEED+k); np.save(f, EMB['scVI'][k])
            except Exception as e:
                print('  scVI failed -> dropped from comparison:', type(e).__name__, str(e)[:200]); RUN_SCVI = False; REPS.remove('scVI'); EMB.pop('scVI', None)
    # --- scGPT adapted / scratch: unlabeled masked-value training on TRAINING patients' cells only ---
    for rep, pre, lr in [('scGPT_adapted', True, ADAPT_LR)] + ([('scGPT_scratch', False, SCRATCH_LR)] if RUN_SCRATCH else []):
        f = CACHE/f'fold{k}_{rep}.npy'
        if f.exists(): EMB[rep][k] = np.load(f); continue
        corpus = adapt_corpus(train_rows, SEED+k)
        torch.manual_seed(SEED+k); m, _ = build_scgpt(SCGPT_DIR, pretrained=pre)
        ADAPT_LOG[f'{rep}_fold{k}'] = adapt_scgpt(m, X_V[corpus], GENE_IDS, VOCAB, CFG, ADAPT_STEPS, lr, SEED+k, f'{rep}/fold{k}')
        EMB[rep][k] = embed_cells(m, X_V[PROG_IDX], GENE_IDS, VOCAB, CFG, seed=SEED); np.save(f, EMB[rep][k])
        del m
        if DEVICE == 'cuda': torch.cuda.empty_cache()
json.dump({k: {a: b for a, b in v.items() if a != 'train_loss'} for k, v in ADAPT_LOG.items()}, open(OUT/'02_adaptation_log.json', 'w'), indent=1)
print('\nrepresentations ready:', REPS)

In [ ]:
#@title 10 · Identical linear probe per representation; operating point fixed on TRAINING patients (nested, patient-grouped)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

po = obs.iloc[PROG_IDX].reset_index(drop=True)      # row i of every embedding == po.iloc[i]
y_all = po.y.to_numpy().astype(int)

def capped(idx, seed):
    r = np.random.default_rng(seed); keep = []
    for _, g in po.iloc[idx].groupby(['sample', 'y']).indices.items():
        keep.extend(idx[g][r.permutation(len(g))[:PROBE_MAX_PER_SAMPLE_LABEL]])
    return np.sort(np.array(keep))

def lr(C): return LogisticRegression(C=C, class_weight='balanced', max_iter=3000)

def probe_fold(E, k, seed=SEED):
    te = po.patient.isin(FOLDS[k]).to_numpy(); tr_idx = capped(np.flatnonzero(~te), seed+k); te_idx = np.flatnonzero(te)
    sc = StandardScaler().fit(E[tr_idx]); A = sc.transform(E[tr_idx]); yt = y_all[tr_idx]; g = po.patient.to_numpy()[tr_idx]
    best = (-1, None, None)
    for C in C_GRID:                                   # inner patient-grouped CV picks C and yields out-of-fold training scores
        oof = np.full(len(A), np.nan)
        for a, b in GroupKFold(n_splits=3).split(A, yt, g):
            if len(np.unique(yt[a])) < 2: continue
            oof[b] = lr(C).fit(A[a], yt[a]).decision_function(A[b])
        ok = ~np.isnan(oof)
        auc = roc_auc_score(yt[ok], oof[ok]) if ok.sum() and len(np.unique(yt[ok])) == 2 else -1
        if auc > best[0]: best = (auc, C, oof)
    auc_in, C, oof = best
    thr = np.quantile(oof[(yt == 1) & ~np.isnan(oof)], 1 - TARGET_TPR)     # threshold giving TARGET_TPR on training malignant cells
    s = lr(C).fit(A, yt).decision_function(sc.transform(E[te_idx]))
    return pd.DataFrame(dict(row=te_idx, rep=None, fold=k, score=s, call=(s >= thr).astype(int), C=C, inner_auc=auc_in, thr=thr))

parts = []
for rep in REPS:
    for k in range(N_FOLDS):
        d = probe_fold(EMB[rep][k], k); d['rep'] = rep; parts.append(d)
PRED = pd.concat(parts, ignore_index=True)
meta = po[['cell', 'patient', 'sample', 'stage', 'y', 'geno', 'CellType']].copy(); meta['row'] = np.arange(len(po))
PRED = PRED.merge(meta, on='row'); PRED.to_csv(OUT/'03_cell_level_predictions.csv.gz', index=False)
display(PRED.groupby('rep')[['inner_auc']].mean().round(3).rename(columns={'inner_auc': 'mean inner-CV AUROC (training patients)'}))

In [ ]:
#@title 11 · False-positive rate at the fixed operating point, with patient-cluster bootstrap and pre-specified contrasts
STRATA = {   # false-positive strata (true label = non-malignant)
    'post_normal':      lambda d: (d.stage == 'post') & (d.y == 0),
    'post_normal_WTcov':lambda d: (d.stage == 'post') & (d.y == 0) & (d.geno == 'WT-only'),
    'dx_normal':        lambda d: (d.stage == 'dx') & (d.y == 0),
    'healthy':          lambda d: (d.stage == 'healthy') & (d.y == 0)}
SENS = {     # sensitivity strata (true label = malignant)
    'mal_all':      lambda d: d.y == 1,
    'mal_post':     lambda d: (d.stage == 'post') & (d.y == 1),
    'mal_MUT':      lambda d: (d.y == 1) & (d.geno == 'MUT')}
ALLS = {**STRATA, **SENS}
rng = np.random.default_rng(SEED)

def patient_rates(rep, name):
    d = PRED[PRED.rep == rep]; d = d[ALLS[name](d)]
    g = d.groupby('patient').call.agg(['mean', 'size']); return g[g['size'] >= MIN_CELLS_PATIENT_STRATUM]['mean']

rows, BOOT = [], {}
for name in ALLS:
    pats = sorted(set.intersection(*[set(patient_rates(r, name).index) for r in REPS]))
    if not pats: continue
    idx = rng.integers(0, len(pats), size=(N_BOOT, len(pats)))          # SAME patient resamples for every representation (paired)
    for rep in REPS:
        v = patient_rates(rep, name).reindex(pats).to_numpy(); b = v[idx].mean(1); BOOT[(rep, name)] = b
        d = PRED[PRED.rep == rep]; d = d[ALLS[name](d)]
        rows.append(dict(stratum=name, rep=rep, n_patients=len(pats), n_cells=int(len(d)), macro_rate=v.mean(),
                         ci_lo=np.percentile(b, 2.5), ci_hi=np.percentile(b, 97.5), pooled_rate=d.call.mean()))
RES = pd.DataFrame(rows); RES.to_csv(OUT/'04_rates_by_stratum.csv', index=False)
print('call rate by stratum.  FALSE-POSITIVE strata want LOW; SENSITIVITY strata (mal_*) should sit near', TARGET_TPR)
display(RES.pivot(index='rep', columns='stratum', values='macro_rate').reindex(REPS).round(3))
display(RES[['stratum', 'rep', 'n_patients', 'n_cells']].pivot(index='stratum', columns='rep', values='n_patients'))

# ---- pre-specified contrasts on the PRIMARY stratum (post-treatment normal progenitors) ----
def contrast(a, b, name='post_normal'):
    if (a, name) not in BOOT or (b, name) not in BOOT: return None
    d = BOOT[(a, name)] - BOOT[(b, name)]
    pt = RES[(RES.rep == a) & (RES.stratum == name)].macro_rate.iloc[0] - RES[(RES.rep == b) & (RES.stratum == name)].macro_rate.iloc[0]
    return pt, np.percentile(d, 2.5), np.percentile(d, 97.5)
def tpr_gap(a, b):
    ra, rb = (RES[(RES.rep == x) & (RES.stratum == 'mal_all')].macro_rate.iloc[0] for x in (a, b)); return ra - rb
CONTRASTS = [('scGPT_adapted', 'scGPT_zeroshot', 'C1 adaptation helps (vs original scGPT)'),
             ('scGPT_adapted', 'PCA',            'C2 beats PCA'),
             ('scGPT_adapted', 'scVI',           'C3 beats scVI')] + ([('scGPT_adapted', 'scGPT_scratch', 'C4 pretraining matters (vs scratch)')] if RUN_SCRATCH else [])
crow = []
for a, b, lab in CONTRASTS:
    c = contrast(a, b)
    if c is None: continue
    crow.append(dict(contrast=lab, FPR_difference=c[0], ci_lo=c[1], ci_hi=c[2], sensitivity_gap=tpr_gap(a, b),
                     FPR_lower_CI_excludes_0=c[2] < 0, sensitivity_matched_within_5pts=abs(tpr_gap(a, b)) <= 0.05))
CT = pd.DataFrame(crow); CT.to_csv(OUT/'05_primary_contrasts.csv', index=False); display(CT.round(3))

In [ ]:
#@title 12 · Figures: false-positive rate by representation; per-patient view
import matplotlib.pyplot as plt
OKABE = {'PCA': '#56B4E9', 'scVI': '#009E73', 'scGPT_zeroshot': '#999999', 'scGPT_adapted': '#D55E00', 'scGPT_scratch': '#CC79A7'}
LABEL = {'PCA': 'HVG PCA', 'scVI': 'scVI', 'scGPT_zeroshot': 'scGPT original', 'scGPT_adapted': 'scGPT AML-adapted', 'scGPT_scratch': 'scGPT from scratch'}
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
for ax, name, title in zip(axes, ['post_normal', 'dx_normal', 'healthy'],
                           ['Post-treatment normal progenitors\n(PRIMARY)', 'Diagnosis-sample normal progenitors', 'Healthy-donor progenitors']):
    sub = RES[RES.stratum == name].set_index('rep').reindex(REPS)
    for i, rep in enumerate(REPS):
        r = sub.loc[rep]; ax.errorbar(i, r.macro_rate, yerr=[[r.macro_rate - r.ci_lo], [r.ci_hi - r.macro_rate]], fmt='o', ms=8,
                                      color=OKABE[rep], mec='white', mew=1.5, capsize=3, lw=1.5)
    ax.set_xticks(range(len(REPS))); ax.set_xticklabels([LABEL[r] for r in REPS], rotation=30, ha='right')
    ax.set_title(f'{title}\nn={int(sub.n_patients.iloc[0])} patients, {int(sub.n_cells.iloc[0])} cells', fontsize=10, loc='left')
axes[0].set_ylabel(f'False-positive rate at ~{int(TARGET_TPR*100)}% training sensitivity\n(patient-macro mean, 95% patient bootstrap CI)')
axes[0].set_ylim(0, 1); fig.tight_layout(); fig.savefig(FIG/'01_fpr_by_stratum.png', dpi=200); plt.show()

pp = PRED[PRED.stage.eq('post') & PRED.y.eq(0)].groupby(['patient', 'rep']).call.agg(['mean', 'size']).reset_index()
pp = pp[pp['size'] >= MIN_CELLS_PATIENT_STRATUM]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for p, g in pp.groupby('patient'):
    g = g.set_index('rep').reindex(REPS); ax.plot(range(len(REPS)), g['mean'], color='#BBBBBB', lw=1, zorder=1)
    ax.text(len(REPS)-0.9, g['mean'].iloc[-1], f'{p} (n={int(g["size"].iloc[0])})', fontsize=7, va='center', color='#555555')
for i, rep in enumerate(REPS): ax.scatter([i]*len(pp[pp.rep == rep]), pp[pp.rep == rep]['mean'], color=OKABE[rep], s=40, zorder=2, edgecolor='white')
ax.set_xticks(range(len(REPS))); ax.set_xticklabels([LABEL[r] for r in REPS], rotation=30, ha='right'); ax.set_ylim(-0.02, 1.02)
ax.set_ylabel('Fraction of post-treatment normal progenitors\ncalled malignant'); ax.set_title('Each line is one patient: is the effect shared or driven by one person?', loc='left', fontsize=10)
fig.tight_layout(); fig.savefig(FIG/'02_post_normal_per_patient.png', dpi=200); plt.show()

In [ ]:
#@title 13 · Pre-specified decision rule, save outputs, optional all-patient adapted checkpoint
n_pat = int(RES[(RES.stratum == 'post_normal')].n_patients.iloc[0]); n_cell = int(RES[(RES.stratum == 'post_normal')].n_cells.iloc[0])
crit = {r.contrast: bool(r.FPR_lower_CI_excludes_0 and r.sensitivity_matched_within_5pts) for r in CT.itertuples()}
print('Per-contrast pass = FPR difference CI entirely below 0 AND sensitivity within 5 points of the comparator:')
for k_, v in crit.items(): print(f'  {"PASS" if v else "fail"}  {k_}')
core = [v for k_, v in crit.items() if k_.startswith(('C1', 'C2', 'C3'))]
if len(core) < 3:
    verdict = 'INCOMPLETE: a required comparator (scVI) is missing, so the pre-specified decision rule cannot be applied. Fix scVI and re-run.'
elif not PRIMARY_CONFIRMATORY:
    verdict = (f'UNDERPOWERED / EXPLORATORY: primary stratum has {n_pat} patients, {n_cell} cells (gate not met). '
               'Even if C1-C3 pass, this supports only a pilot signal, not a model-development claim.')
elif all(core):
    verdict = 'GO: adaptation lowered post-treatment false positives vs original scGPT, PCA and scVI at matched sensitivity. Next: external cohort with independent labels.'
else:
    verdict = 'NO-GO: adaptation did not beat all of original scGPT, PCA and scVI on the primary endpoint. Per the decision rule fixed in this notebook before any real result, abandon the model-development claim.'
print('\nVERDICT:', verdict)
(OUT/'README_RESULTS.md').write_text(f'''# AML-adapted scGPT false-positive benchmark (van Galen GSE116256)

**Verdict:** {verdict}

Primary stratum: {n_pat} patients / {n_cell} cells. Labels are van Galen classifier calls; genotype-supported subsets are small
(see 00_feasibility_gate.csv). Not a clinical MRD assay; no sensitivity claim for clinical MRD.

Files: 04_rates_by_stratum.csv, 05_primary_contrasts.csv, 03_cell_level_predictions.csv.gz, 02_adaptation_log.json, figures/.
''')
print((OUT/'README_RESULTS.md').read_text())

if TRAIN_FINAL_MODEL:
    f = MODEL/'scGPT_AML_adapted_allpatients.pt'
    if not f.exists():
        corpus = adapt_corpus(np.arange(adata.n_obs), SEED+999)
        torch.manual_seed(SEED); m, _ = build_scgpt(SCGPT_DIR, pretrained=True)
        adapt_scgpt(m, X_V[corpus], GENE_IDS, VOCAB, CFG, ADAPT_STEPS, ADAPT_LR, SEED+999, 'final/all-patients')
        torch.save(m.state_dict(), f)
    print('all-patient adapted checkpoint (for the seven-patient follow-up):', f)
    print('NOTE: this model has seen every van Galen patient. Never evaluate it on van Galen; use it only on independent cohorts.')